# Seebeck dataset audit: what comes with Seebeck, and how much is usable?

Checks **Starrydata2, ACS single-band-model compilation, teMatDb / teMatDb272, ChemDataExtractor TE database and ESTM**.

For every dataset it reports:
1. **Co-occurrence**: of the records that have a Seebeck value, the % that also have σ/ρ, κ (total/lattice/electronic), PF, ZT, carrier concentration, mobility, effective mass, band gap, doping, …
2. **Usability tiers**

| Tier | Needs | Good for |
|---|---|---|
| T0 | Seebeck value | – |
| T1 | + temperature + composition | Seebeck-only ML |
| PF_ready | T1 + σ or ρ | power-factor models |
| ZT_ready | PF + κ | full ZT models |
| SPB_ready | T1 + carrier concentration | single-parabolic-band fitting |
| SPB_full | SPB + mobility | weighted mobility / SPB |
| CLEAN | T1 with no suspicious values or duplicates | a realistic "usable" count |

3. **Checks**: |S| > 1000 µV/K, S = 0, T outside 2–1500 K, duplicates, units, and whether reported ZT matches S²σT/κ.

**How to run:** Runtime → Run all. Put your files in Google Drive (Step 2) or upload them when asked.

## Step 1 · Setup

In [ ]:
!pip -q install openpyxl pyarrow

import json, re, sys, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)

## Step 2 · Get the data into Colab

Choose **one** way:

* **`"drive"`**: put each dataset in its own folder in Google Drive, e.g. `MyDrive/TE_datasets/ESTM/`. Files can be CSV, TSV, Excel, JSON, parquet or **.zip**.
* **`"upload"`**: you will get one upload box per dataset. Press **Cancel** to skip one.

The folder names must match the keys in `DATASETS` below.

In [ ]:
MODE = "upload"                                 # "upload" or "drive"
DRIVE_BASE = "/content/drive/MyDrive/TE_datasets"   # used when MODE == "drive"

# Folder name -> optional overrides (leave {} for automatic detection).
# Overrides you can add if detection is wrong:
#   "layout": "wide" | "long" | "curve"
#   "prop_col": "...", "value_col": "...", "unit_col": "...", "group_cols": ["..."], "sheet": "..."
#   "col_map": {"exact column name": "seebeck"}   # force a column's meaning
DATASETS = {
    "Starrydata2":          {},   # use the curves CSV (+ samples CSV) from the release zip
    "ACS_SPB":              {},   # SI Excel/CSV of the single-band-model compilation
    "teMatDb":              {},   # teMatDb / teMatDb272 export(s)
    "ChemDataExtractor_TE": {},   # CDE thermoelectric DB (CSV or JSON)
    "ESTM":                 {},   # ESTM CSV/Excel
}

In [ ]:
WORK = Path("/content/te_work"); WORK.mkdir(exist_ok=True)

if MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for name in DATASETS:
        DATASETS[name]["path"] = str(Path(DRIVE_BASE) / name)
else:
    from google.colab import files
    for name in DATASETS:
        dest = Path("/content/uploads") / name; dest.mkdir(parents=True, exist_ok=True)
        print(f"\n>>> Upload file(s) for {name}  (Cancel to skip)")
        try:
            up = files.upload()
        except Exception:
            up = {}
        for fname, data in up.items():
            (dest / fname).write_bytes(data)
            Path(fname).unlink(missing_ok=True)
        DATASETS[name]["path"] = str(dest)

# Unzip any .zip into a working folder; don't write into Drive
for name, cfg in list(DATASETS.items()):
    p = Path(cfg["path"])
    if not p.exists() or (p.is_dir() and not any(p.iterdir())):
        print(f"[skip] {name}: nothing at {p}")
        DATASETS.pop(name); continue
    zips = [p] if p.suffix == ".zip" else list(p.rglob("*.zip")) if p.is_dir() else []
    paths = [str(p)] if p.suffix != ".zip" else []
    for z in zips:
        out = WORK / name / z.stem
        if not out.exists():
            with zipfile.ZipFile(z) as zf:
                zf.extractall(out)
        paths.append(str(out))
        print(f"[unzip] {name}: {z.name} -> {out}")
    cfg["path"] = paths
    print(f"[ok] {name}: {paths}")

## Step 3 · Library (just run it)

In [ ]:
OUT_DIR = Path('/content/seebeck_audit_out')
S_MAX_UV = 1000.0          # |S| above this (uV/K) is flagged
T_RANGE = (2.0, 1500.0)    # plausible temperatures (K)

# --------------------------------------------------------------------------------------
# 2. Property vocabulary. ORDER MATTERS: specific patterns are checked before generic ones.
# --------------------------------------------------------------------------------------
PROPERTY_PATTERNS: list[tuple[str, list[str]]] = [
    ("lattice_thermal_cond",   [r"lattice.*(thermal|conduct|kappa)", r"kappa_?l\b", r"κ_?l\b", r"\bkl\b"]),
    ("electronic_thermal_cond", [r"electronic.*(thermal|conduct|kappa)", r"kappa_?e\b", r"κ_?e\b"]),
    ("thermal_cond",   [r"thermal.?conduct", r"\bkappa\b", r"^κ", r"\btc\b", r"^k\s*\(w", r"thermal_?cond"]),
    ("thermal_diffusivity", [r"diffusivity"]),
    ("heat_capacity",  [r"heat.?capacity", r"specific.?heat", r"\bcp\b"]),
    ("seebeck",        [r"seebeck", r"thermopower", r"thermoelectric.?power", r"^alpha\b", r"^s\s*[\(\[_]",
                        r"^s$", r"^seeb"]),
    ("power_factor",   [r"power.?factor", r"^pf\b"]),
    ("zt",             [r"\bzt\b", r"figure.?of.?merit", r"^z\s*t", r"^zt"]),
    ("resistivity",    [r"resistiv", r"^rho\b", r"^ρ", r"^r\s*\(.*ohm"]),
    ("hall_coefficient", [r"hall.?coeff", r"^r_?h\b"]),
    ("carrier_conc",   [r"carrier.?(conc|dens)", r"hall.?(conc|dens|carrier)", r"^n_?h\b", r"^n\s*\(.*cm",
                        r"^p_?h\b", r"^p\s*\(.*cm", r"concentration"]),
    ("mobility",       [r"mobility", r"^mu\b", r"^μ"]),
    ("elec_cond",      [r"electrical.?conduct", r"^sigma\b", r"^σ", r"elec_?cond", r"^conductivity$",
                        r"electric.*conduct"]),
    ("effective_mass", [r"effective.?mass", r"m\*", r"m_?d\b", r"dos.?mass"]),
    ("weighted_mobility", [r"weighted.?mobility", r"mu_?w"]),
    ("band_gap",       [r"band.?gap", r"^e_?g\b", r"^gap"]),
    ("density",        [r"^density", r"relative.?density"]),
    ("temperature",    [r"^temp", r"temperature", r"^t$", r"^t\s*[\(\[_]", r"^t_?k$"]),
    ("composition",    [r"formula", r"composition", r"compound", r"^material", r"^name$", r"^chemical",
                        r"^sample.?name", r"^system"]),
    ("sample_id",      [r"^sample.?id", r"^sid$", r"^sampleid", r"^sample$", r"^sample_?no"]),
    ("doi",            [r"doi", r"^reference", r"^ref$", r"^paper", r"^source", r"^citation", r"^journal"]),
    ("doping",         [r"dopant", r"doping"]),
    ("synthesis",      [r"synthes", r"method", r"process"]),
]
PHYSICAL = ["seebeck", "elec_cond", "resistivity", "thermal_cond", "lattice_thermal_cond",
            "electronic_thermal_cond", "power_factor", "zt", "carrier_conc", "mobility",
            "hall_coefficient", "effective_mass", "weighted_mobility", "band_gap",
            "thermal_diffusivity", "heat_capacity", "density", "temperature", "doping", "synthesis"]

NUM_RE = re.compile(r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?")


def norm(s) -> str:
    return re.sub(r"\s+", " ", str(s)).strip().lower()


def classify(name) -> str | None:
    n = norm(name)
    if not n or n == "nan":
        return None
    for canon, pats in PROPERTY_PATTERNS:
        if any(re.search(p, n) for p in pats):
            return canon
    return None


def to_num(x) -> float:
    """First number in a cell: '150 ± 5' -> 150, '[1,2,3]' -> 1, '2.1e19' -> 2.1e19."""
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    m = NUM_RE.search(str(x).replace("−", "-").replace(",", " ") if "[" not in str(x) else str(x))
    return float(m.group()) if m else np.nan


def array_len(x) -> int:
    """Number of points in a digitized array cell like '[1, 2, 3]' or '1;2;3'."""
    if pd.isna(x):
        return 0
    return len(NUM_RE.findall(str(x)))


# --------------------------------------------------------------------------------------
# 3. Loading
# --------------------------------------------------------------------------------------
TABULAR = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".json", ".jsonl", ".parquet"}


def load_file(p: Path, sheet=None) -> list[pd.DataFrame]:
    ext = p.suffix.lower()
    try:
        if ext in {".csv", ".txt"}:
            d = pd.read_csv(p, low_memory=False, encoding_errors="replace")      # fast C parser
            if d.shape[1] == 1:                                                # ; or tab separated
                d = pd.read_csv(p, sep=None, engine="python", encoding_errors="replace")
            return [d]
        if ext == ".tsv":
            return [pd.read_csv(p, sep="\t", encoding_errors="replace")]
        if ext in {".xlsx", ".xls"}:
            sheets = pd.read_excel(p, sheet_name=sheet if sheet else None)
            if isinstance(sheets, pd.DataFrame):
                sheets = {sheet: sheets}
            out = []
            for name, d in sheets.items():
                d = fix_header(d)
                d["__sheet"] = name
                out.append(d)
            return out
        if ext == ".parquet":
            return [pd.read_parquet(p)]
        if ext == ".jsonl":
            return [pd.json_normalize([json.loads(l) for l in p.open(encoding="utf-8") if l.strip()])]
        if ext == ".json":
            obj = json.loads(p.read_text(encoding="utf-8"))
            if isinstance(obj, dict):  # {"records": [...]} or {id: record}
                lists = [v for v in obj.values() if isinstance(v, list)]
                obj = lists[0] if lists else list(obj.values())
            return [pd.json_normalize(obj)]
    except Exception as e:  # keep going on bad files
        print(f"   ! could not read {p.name}: {e}")
    return []


def fix_header(d: pd.DataFrame) -> pd.DataFrame:
    """SI spreadsheets often have title rows above the header; pick the row with most property hits."""
    if sum(classify(c) is not None for c in d.columns) >= 2:
        return d
    best, best_hits = None, 1
    for i in range(min(15, len(d))):
        hits = sum(classify(v) is not None for v in d.iloc[i].values)
        if hits > best_hits:
            best, best_hits = i, hits
    if best is None:
        return d
    new = d.iloc[best + 1:].copy()
    new.columns = [str(c) for c in d.iloc[best].values]
    return new.reset_index(drop=True)


def load_dataset(path, sheet=None) -> pd.DataFrame:
    """path: a file, a folder (searched recursively), or a list of files/folders."""
    files = []
    for one in (path if isinstance(path, (list, tuple)) else [path]):
        p = Path(one).expanduser()
        if not p.exists():
            raise FileNotFoundError(p)
        files += [p] if p.is_file() else sorted(f for f in p.rglob("*") if f.suffix.lower() in TABULAR)
    frames = []
    for f in files:
        for d in load_file(f, sheet):
            if len(d):
                d["__file"] = f.stem
                frames.append(d)
    if not frames:
        raise ValueError(f"No readable tabular files under {p}")
    print(f"   loaded {len(files)} file(s), {sum(len(f) for f in frames):,} rows")
    return pd.concat(frames, ignore_index=True, sort=False)


# --------------------------------------------------------------------------------------
# 4. Layout detection
# --------------------------------------------------------------------------------------
def find_col(df, patterns, exclude=()) -> str | None:
    for c in df.columns:
        if c in exclude:
            continue
        n = norm(c)
        if any(re.search(p, n) for p in patterns):
            return c
    return None


def property_column(df) -> str | None:
    """A text column whose *values* are property names (Seebeck, ZT, ...)."""
    best, best_score = None, 0
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        vals = df[c].dropna().astype(str)
        if vals.empty or vals.nunique() > 2000:
            continue
        sample = vals.drop_duplicates().head(500)
        kinds = sample.map(classify)
        phys = kinds.isin(PHYSICAL).sum()
        has_s = (kinds == "seebeck").any()
        score = phys / len(sample) + (1 if has_s else 0) + (0.5 if re.search(r"prop|quantity|param|type", norm(c)) else 0)
        if has_s and score > best_score:
            best, best_score = c, score
    return best


def detect_layout(df, cfg) -> str:
    if cfg.get("layout", "auto") != "auto":
        return cfg["layout"]
    xcol = find_col(df, [r"prop.*_?x$", r"propertyname_?x", r"x_?prop", r"^x_?name"])
    ycol = find_col(df, [r"prop.*_?y$", r"propertyname_?y", r"y_?prop", r"^y_?name"])
    if xcol and ycol:
        return "curve"
    wide_hits = {classify(c) for c in df.columns} & set(PHYSICAL)
    if "seebeck" in wide_hits:
        return "wide"
    if property_column(df):
        return "long"
    return "wide"


# --------------------------------------------------------------------------------------
# 5. Build a standard table: one row per record with canonical property columns
#    presence  -> bool per property   values -> first numeric value per property
# --------------------------------------------------------------------------------------
def standardize_wide(df, cfg):
    col_map = {}
    for c in df.columns:
        if str(c).startswith("__"):
            continue
        k = cfg.get("col_map", {}).get(c) or classify(c)
        if k and k not in col_map.values():   # first matching column wins
            col_map[c] = k
    vals = pd.DataFrame(index=df.index)
    for c, k in col_map.items():
        if k in ("composition", "sample_id", "doi", "doping", "synthesis"):
            vals[k] = df[c].where(df[c].astype(str).str.strip().ne("") & df[c].notna())
        else:
            vals[k] = df[c].map(to_num)
    units = {k: extract_unit(c) for c, k in col_map.items()}
    return vals, col_map, units, None


def extract_unit(colname) -> str:
    m = re.search(r"[\(\[](.+?)[\)\]]", str(colname))
    return m.group(1) if m else ""


def g_first(d, keys, prop, val):
    """Material x property table holding the first numeric value (works on pandas 1.x-3.x)."""
    t = d.groupby(list(keys) + [prop], dropna=False, sort=True)[val].first().unstack(prop)
    t.columns.name = None
    return t


def standardize_long(df, cfg):
    pcol = cfg.get("prop_col") or property_column(df)
    if pcol is None:
        raise ValueError("Could not find the property-name column; pass --prop-col")
    vcol = cfg.get("value_col") or find_col(df, [r"^value", r"^val$", r"^raw_?value", r"^y$", r"^data$",
                                                 r"^magnitude", r"^number"], exclude=(pcol,))
    if vcol is None:  # fall back to the most numeric-looking column
        numeric_share = {c: df[c].map(to_num).notna().mean() for c in df.columns if c != pcol}
        vcol = max(numeric_share, key=numeric_share.get)
    ucol = cfg.get("unit_col") or find_col(df, [r"unit"], exclude=(pcol, vcol))
    tcol = find_col(df, [r"^temp", r"temperature", r"^t$", r"^t\s*[\(\[_]"], exclude=(pcol, vcol))
    comp = find_col(df, dict(PROPERTY_PATTERNS)["composition"], exclude=(pcol, vcol))
    sid = find_col(df, dict(PROPERTY_PATTERNS)["sample_id"], exclude=(pcol, vcol))
    doi = find_col(df, dict(PROPERTY_PATTERNS)["doi"], exclude=(pcol, vcol))

    d = df.copy()
    d["__prop"] = d[pcol].map(classify)
    d["__val"] = d[vcol].map(to_num)
    if tcol:
        d["__T"] = d[tcol].map(to_num)
    group = cfg.get("group_cols") or [c for c in (sid, comp, doi) if c] or ["__file"]
    print(f"   long layout: property='{pcol}' value='{vcol}' unit='{ucol}' T='{tcol}' group={group}")

    unmatched = d.loc[d["__prop"].isna(), pcol].value_counts().head(15)
    if len(unmatched):
        print("   property names NOT mapped (add to PROPERTY_PATTERNS or col_map if relevant):")
        for name, n in unmatched.items():
            print(f"      {n:>7,}  {name}")

    # material-level view: which properties does each material/paper have at all?
    g = d.dropna(subset=["__prop"]).groupby(group, dropna=False)
    vals = g_first(d.dropna(subset=["__prop"]), group, "__prop", "__val").reset_index()
    if tcol and "temperature" not in vals.columns:
        has_t = g["__T"].apply(lambda s: s.notna().any()).reset_index(drop=True)
        vals["temperature"] = np.where(has_t.values, 1.0, np.nan)
    if comp:
        vals["composition"] = vals[comp] if comp in vals else np.nan
    if doi and doi in vals:
        vals["doi"] = vals[doi]

    # point-level view: same material AND same temperature (stricter; used for ZT consistency)
    point = None
    if tcol:
        point = g_first(d.dropna(subset=["__prop", "__T"]), group + ["__T"], "__prop", "__val").reset_index().rename(columns={"__T": "temperature"})
        if comp:
            point["composition"] = point[comp]

    units = {}
    if ucol:
        for k in ("seebeck", "elec_cond", "resistivity", "thermal_cond", "carrier_conc"):
            u = d.loc[d["__prop"] == k, ucol].astype(str).value_counts().head(3)
            units[k] = "; ".join(f"{i} ({n})" for i, n in u.items())
    col_map = {pcol: "(property names)", vcol: "(values)"}
    return vals, col_map, units, point


def standardize_curve(df, cfg):
    xcol = find_col(df, [r"prop.*_?x$", r"propertyname_?x", r"x_?prop", r"^x_?name"])
    ycol = find_col(df, [r"prop.*_?y$", r"propertyname_?y", r"y_?prop", r"^y_?name"])
    ydat = find_col(df, [r"^y$", r"^y_?values?$", r"^y_?data"], exclude=(ycol,))
    sid = cfg.get("group_cols") or [find_col(df, [r"^sample.?id", r"^sid$", r"^sampleid", r"sample"]) or "__file"]
    uy = find_col(df, [r"unit.*_?y$", r"y_?unit"])
    print(f"   curve layout: x-prop='{xcol}' y-prop='{ycol}' y-data='{ydat}' sample={sid}")

    d = df.copy()
    d["__y"] = d[ycol].map(classify)
    d["__x"] = d[xcol].map(classify)
    d["__npts"] = d[ydat].map(array_len) if ydat else 1
    d["__yfirst"] = d[ydat].map(to_num) if ydat else 1.0

    unmatched = d.loc[d["__y"].isna(), ycol].value_counts().head(15)
    if len(unmatched):
        print("   y-property names NOT mapped:")
        for name, n in unmatched.items():
            print(f"      {n:>7,}  {name}")

    vals = g_first(d.dropna(subset=["__y"]), sid, "__y", "__yfirst")
    # temperature available if Seebeck is plotted vs T
    s_vs_t = d[(d["__y"] == "seebeck") & (d["__x"] == "temperature")].groupby(sid)["__npts"].sum()
    vals["temperature"] = s_vs_t.reindex(vals.index).where(lambda s: s > 0).map(lambda _: 1.0)
    vals["__seebeck_points_vs_T"] = s_vs_t.reindex(vals.index)
    comp = find_col(df, dict(PROPERTY_PATTERNS)["composition"])
    if comp:
        vals["composition"] = df.groupby(sid)[comp].first().reindex(vals.index)
    else:
        vals["composition"] = "assumed"  # Starrydata composition lives in samples.csv; join it for real checks
        print("   (no composition column in curves file -> join the samples table for composition)")
    vals = vals.reset_index()
    vals["sample_id"] = vals[sid[0]]
    units = {}
    if uy:
        u = d.loc[d["__y"] == "seebeck", uy].astype(str).value_counts().head(3)
        units["seebeck"] = "; ".join(f"{i} ({n})" for i, n in u.items())
    total_pts = int(d.loc[d["__y"] == "seebeck", "__npts"].sum())
    print(f"   Seebeck curves: {int((d['__y'] == 'seebeck').sum()):,}  |  digitized Seebeck points: {total_pts:,}")
    return vals, {xcol: "(x property)", ycol: "(y property)"}, units, None


# --------------------------------------------------------------------------------------
# 6. Analysis
# --------------------------------------------------------------------------------------
def present(vals, k) -> pd.Series:
    if k not in vals.columns:
        return pd.Series(False, index=vals.index)
    return vals[k].notna()


def seebeck_in_uV(s: pd.Series) -> pd.Series:
    """Guess unit from magnitude: median |S| < 0.01 -> V/K, < 10 -> mV/K, else uV/K."""
    med = s.abs().median()
    if pd.isna(med):
        return s
    if med < 0.01:
        return s * 1e6
    if med < 10:
        return s * 1e3
    return s


def analyse(name, vals, point, units, col_map):
    n_total = len(vals)
    has_s = present(vals, "seebeck")
    sv = vals[has_s]
    res = {"dataset": name, "records": n_total, "with_seebeck": int(has_s.sum())}

    # co-occurrence among Seebeck records
    cooc = {}
    for k in PHYSICAL + ["composition", "doi"]:
        if k == "seebeck" or k not in vals.columns:
            continue
        cooc[k] = present(sv, k).mean() * 100 if len(sv) else 0.0
    cooc = pd.Series(cooc).sort_values(ascending=False)

    # tiers
    comp_ok = present(sv, "composition") | present(sv, "sample_id")
    t_ok = present(sv, "temperature")
    elec = present(sv, "elec_cond") | present(sv, "resistivity")
    kap = present(sv, "thermal_cond") | (present(sv, "lattice_thermal_cond") & present(sv, "electronic_thermal_cond"))
    T1 = comp_ok & t_ok
    tiers = {
        "T0_seebeck": len(sv),
        "T1_+T+composition": int(T1.sum()),
        "PF_ready(+sigma/rho)": int((T1 & elec).sum()),
        "ZT_ready(+kappa)": int((T1 & elec & kap).sum()),
        "SPB_ready(+carrier_conc)": int((T1 & present(sv, "carrier_conc")).sum()),
        "SPB_full(+n,+mobility)": int((T1 & present(sv, "carrier_conc") & present(sv, "mobility")).sum()),
    }

    # sanity on Seebeck values
    s_uv = seebeck_in_uV(sv["seebeck"].astype(float))
    bad_s = s_uv.abs() > S_MAX_UV
    zero_s = s_uv == 0
    if "temperature" in sv and sv["temperature"].dropna().nunique() > 2:
        tv = sv["temperature"].astype(float)
        bad_t = tv.notna() & ((tv < T_RANGE[0]) | (tv > T_RANGE[1]))
    else:
        bad_t = pd.Series(False, index=sv.index)
    key = [c for c in ("sample_id", "composition", "temperature", "seebeck", "doi") if c in sv.columns]
    dup = sv.duplicated(subset=key, keep="first") if key else pd.Series(False, index=sv.index)
    clean = T1 & ~bad_s & ~zero_s & ~bad_t & ~dup
    tiers["CLEAN(T1+sane+unique)"] = int(clean.sum())
    res.update(tiers)
    res.update({
        "S_|S|>1000uV/K": int(bad_s.sum()),
        "S_zero": int(zero_s.sum()),
        "T_out_of_range": int(bad_t.sum()),
        "duplicates": int(dup.sum()),
        "S_median_uV/K": round(float(s_uv.median()), 1) if len(s_uv) else np.nan,
        "S_p1_p99_uV/K": f"{s_uv.quantile(.01):.0f} .. {s_uv.quantile(.99):.0f}" if len(s_uv) else "",
        "n_type_%": round(float((s_uv < 0).mean() * 100), 1) if len(s_uv) else np.nan,
        "unique_compositions": int(sv["composition"].nunique()) if "composition" in sv else np.nan,
        "unique_sources": int(sv["doi"].nunique()) if "doi" in sv else np.nan,
    })

    # ZT consistency check where S, sigma/rho, kappa, ZT and T all exist at the same point
    zt_tbl = point if point is not None else vals
    res.update(zt_consistency(zt_tbl))

    # print report
    print(f"\n   Records: {n_total:,}   with Seebeck: {len(sv):,} ({len(sv) / max(n_total, 1):.0%})")
    if units:
        print("   Units seen:", {k: v for k, v in units.items() if v})
    print("   Properties co-occurring with Seebeck (% of Seebeck records):")
    for k, v in cooc.items():
        bar = "#" * int(v / 4)
        print(f"      {k:<26}{v:6.1f}%  {bar}")
    print("   Usability tiers:")
    for k, v in tiers.items():
        print(f"      {k:<28}{v:>9,}  ({v / max(len(sv), 1):.0%})")
    print(f"   Sanity: |S|>{S_MAX_UV:.0f} uV/K: {res['S_|S|>1000uV/K']:,}   S=0: {res['S_zero']:,}   "
          f"T out of range: {res['T_out_of_range']:,}   duplicates: {res['duplicates']:,}")
    print(f"   Seebeck median {res['S_median_uV/K']} uV/K, p1..p99 {res['S_p1_p99_uV/K']}, n-type {res['n_type_%']}%")
    if res.get("zt_check_rows"):
        print(f"   ZT check: {res['zt_check_rows']:,} points, {res['zt_consistent_%']}% within ±30% "
              f"(sigma unit guess: {res['zt_sigma_unit']})")
    return res, cooc.rename(name)


def zt_consistency(t):
    need = {"seebeck", "thermal_cond", "zt", "temperature"}
    if not need.issubset(t.columns) or not ({"elec_cond", "resistivity"} & set(t.columns)):
        return {}
    d = t.copy()
    if d["temperature"].dropna().nunique() <= 2:   # temperature is only a flag, not a value
        return {}
    d = d.dropna(subset=list(need))
    if "elec_cond" in d and d["elec_cond"].notna().any():
        sig = d["elec_cond"].astype(float)
    else:
        sig = 1.0 / d["resistivity"].astype(float)
    ok = sig.notna() & (sig > 0)
    d, sig = d[ok], sig[ok]
    if len(d) < 5:
        return {}
    S = seebeck_in_uV(d["seebeck"].astype(float)) * 1e-6
    T, kap, zt = d["temperature"].astype(float), d["thermal_cond"].astype(float), d["zt"].astype(float)
    best = None
    # sigma could be S/m, S/cm, or (from rho) 1/(ohm m), 1/(mOhm cm), 1/(uOhm m)... pick the scale that fits best
    for label, f in {"S/m": 1, "S/cm": 100, "1/(mOhm cm)": 1e5, "1/(uOhm m)": 1e6, "1e4 S/m": 1e4}.items():
        ratio = (S ** 2 * sig * f * T / kap) / zt
        med_err = (np.log10(ratio.replace([np.inf, -np.inf], np.nan))).abs().median()
        if best is None or med_err < best[0]:
            best = (med_err, label, ratio)
    _, label, ratio = best
    within = ((ratio > 0.7) & (ratio < 1.3)).mean() * 100
    return {"zt_check_rows": len(d), "zt_consistent_%": round(float(within), 1), "zt_sigma_unit": label}


# --------------------------------------------------------------------------------------
# 7. Driver
# --------------------------------------------------------------------------------------
def run_one(name, cfg):
    print("\n" + "=" * 90 + f"\n{name}\n" + "=" * 90)
    df = load_dataset(cfg["path"], cfg.get("sheet"))
    print(f"   columns ({len(df.columns)}): {list(df.columns)[:40]}{' ...' if len(df.columns) > 40 else ''}")
    layout = detect_layout(df, cfg)
    print(f"   layout: {layout}")
    std = {"wide": standardize_wide, "long": standardize_long, "curve": standardize_curve}[layout]
    vals, col_map, units, point = std(df, cfg)
    if layout == "wide":
        print("   column mapping (check this!):")
        for c, k in col_map.items():
            print(f"      {str(c)[:45]:<47}-> {k}")
        unmapped = [c for c in df.columns if c not in col_map and not str(c).startswith("__")]
        if unmapped:
            print(f"   unmapped columns: {unmapped[:30]}")
    OUT_DIR.mkdir(exist_ok=True)
    pd.Series(col_map, name="canonical").to_csv(OUT_DIR / f"{name}_column_map.csv")
    return analyse(name, vals, point, units, col_map)

print('Library loaded.')

### Optional: look at the raw files first
Set `LOOK_AT` to a dataset name to list its columns and first rows. Use this to fix mappings with `col_map` / `prop_col`.

In [ ]:
LOOK_AT = None      # e.g. "ESTM"

if LOOK_AT:
    for f in [f for p in DATASETS[LOOK_AT]["path"] for f in ([Path(p)] if Path(p).is_file() else Path(p).rglob("*"))]:
        if f.is_file() and f.suffix.lower() in {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".json", ".jsonl", ".parquet"}:
            print("\n#####", f)
            for d in load_file(f):
                print(d.shape); display(d.head())


## Step 4 · Run the audit

In [ ]:
rows, coocs = [], []
for name, cfg in DATASETS.items():
    try:
        r, c = run_one(name, cfg)
        rows.append(r); coocs.append(c)
    except Exception as e:
        print(f"   !! skipped {name}: {type(e).__name__}: {e}")

OUT_DIR.mkdir(exist_ok=True)
summary = pd.DataFrame(rows).set_index("dataset")
cooc = pd.concat(coocs, axis=1).fillna(0).round(1)
summary.to_csv(OUT_DIR / "summary.csv"); cooc.to_csv(OUT_DIR / "cooccurrence.csv")

## Step 5 · Results
### How many Seebeck records are usable at each tier

In [ ]:
tier_cols = [c for c in summary.columns if c.split("_")[0] in ("T0", "T1", "PF", "ZT", "SPB", "CLEAN")]
tiers = summary[["records", "with_seebeck"] + tier_cols].T
display(tiers.style.format("{:,.0f}").background_gradient(axis=1, cmap="Blues"))

pct = summary[tier_cols].div(summary["with_seebeck"].replace(0, np.nan), axis=0) * 100
ax = pct.T.plot.bar(figsize=(10, 4), width=0.8)
ax.set_ylabel("% of Seebeck records"); ax.set_title("Usability tiers"); ax.set_ylim(0, 105)
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

### What comes with Seebeck? (% of Seebeck records that also have each property)

In [ ]:
c = cooc.loc[(cooc > 0).any(axis=1)]
fig, ax = plt.subplots(figsize=(1.6 * len(c.columns) + 3, 0.4 * len(c) + 1.5))
im = ax.imshow(c.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(c.columns))); ax.set_xticklabels(c.columns, rotation=30, ha="right")
ax.set_yticks(range(len(c))); ax.set_yticklabels(c.index)
for i in range(c.shape[0]):
    for j in range(c.shape[1]):
        v = c.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 60 else "black")
fig.colorbar(im, label="%"); ax.set_title("What comes with Seebeck?")
plt.tight_layout(); fig.savefig(OUT_DIR / "cooccurrence_heatmap.png", dpi=150); plt.show()

### Data-quality checks

In [ ]:
qc_cols = [c for c in summary.columns if c not in tier_cols + ["records", "with_seebeck"]]
display(summary[qc_cols].T)

## Step 6 · Download results (summary.csv, cooccurrence.csv, heatmap, column maps)

In [ ]:
shutil.make_archive("/content/seebeck_audit_results", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("/content/seebeck_audit_results.zip")
except Exception:
    print("Saved to /content/seebeck_audit_results.zip")

### Notes
* **Check the column mapping printed in Step 4.** Names are matched by pattern. Anything listed as *NOT mapped* or *unmapped* can be fixed with `col_map` or `prop_col` in `DATASETS`.
* **Starrydata2**: the curves file has no composition column, so composition is treated as present. Join the samples table for a real check.
* **ChemDataExtractor**: records are grouped by compound + DOI, so S, σ and κ may come from different temperatures. The ZT check only compares values at the same temperature.